<a href="https://colab.research.google.com/github/tnkkzk0322/keiba-horse-profile-extractor/blob/main/%E7%AB%B6%E9%A6%AC%E3%83%AA%E3%82%B6%E3%83%AB%E3%83%88%E9%9B%86%E8%A8%88.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# netkeiba race result Google Sheets updater
# Colabで実行する前提です。

# ============================================================
# 1. ライブラリ準備
# ============================================================
try:
    import requests
    import gspread
    from bs4 import BeautifulSoup
except ImportError:
    import sys
    import subprocess
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'requests',
        'gspread',
        'beautifulsoup4',
        'lxml',
    ])
    import requests
    import gspread
    from bs4 import BeautifulSoup

import re
import time
from datetime import datetime, timezone, timedelta

try:
    from google.colab import auth
    from google.auth import default
except Exception:
    auth = None
    default = None

# ============================================================
# 2. ユーザー設定
# ============================================================
SPREADSHEET_URL = 'https://docs.google.com/spreadsheets/d/1Vg6Iol4R6jynQLQh7IJ1JfE9PuRT0jD5l-61VsbcPsk/edit?gid=0#gid=0'

# ここだけ更新してください。
RACE_ID_LIST = [
    '202603020101',  # 中央 福島1R サンプル
    '202654062701',  # 地方 高知1R サンプル
]

DEBUG = True
REQUEST_INTERVAL_SEC = 1.0
JST = timezone(timedelta(hours=9))

SHEET_PREDICTION = '予想'
SHEET_RESULT = '結果'

JRA_PLACE_CODES = {
    '01': '札幌',
    '02': '函館',
    '03': '福島',
    '04': '新潟',
    '05': '東京',
    '06': '中山',
    '07': '中京',
    '08': '京都',
    '09': '阪神',
    '10': '小倉',
}

PREDICTION_RANK_HEADERS = [f'予想{i}着' for i in range(1, 11)]

PREDICTION_HEADERS = [
    'race_id',
    '開催区分',
    '日付',
    '場',
    'R',
    'レース名',
    'コース種別',
    '距離m',
    'コース補足',
    '天候',
    '馬場',
    '頭数',
    *PREDICTION_RANK_HEADERS,
]

RESULT_FINISH_HEADERS = [f'{i}着' for i in range(1, 11)]
PAYOUT_HEADERS_IN_RESULT = [
    '単勝',
    '複勝（1着）',
    '複勝（2着）',
    '複勝（3着）',
    '枠連',
    '馬連',
    'ワイド（1-2着）',
    'ワイド（1-3着）',
    'ワイド（2-3着）',
    '馬単',
    '3連複',
    '3連単',
]

RESULT_HEADERS = [
    'race_id',
    '開催区分',
    '日付',
    '場',
    *RESULT_FINISH_HEADERS,
    *PAYOUT_HEADERS_IN_RESULT,
]

# ============================================================
# 3. 共通ユーティリティ
# ============================================================
def debug_print(message):
    if DEBUG:
        print(f'[DEBUG] {message}')


def warn_print(message):
    print(f'[WARN] {message}')


def clean_text(value):
    if value is None:
        return ''
    text = str(value).replace('\xa0', ' ')
    return re.sub(r'\s+', ' ', text).strip()


def cell_text(element, separator=' '):
    if element is None:
        return ''
    return clean_text(element.get_text(separator, strip=True))


def meta_content(soup, prop):
    tag = soup.find('meta', attrs={'property': prop}) or soup.find('meta', attrs={'name': prop})
    if tag is None:
        return ''
    return clean_text(tag.get('content', ''))


def parse_spreadsheet_id(url):
    match = re.search(r'/spreadsheets/d/([a-zA-Z0-9-_]+)', url)
    if not match:
        raise ValueError('SPREADSHEET_URLからスプレッドシートIDを取得できません。URLを確認してください。')
    return match.group(1)


def normalize_race_ids(race_ids):
    normalized = []
    seen = set()
    for race_id in race_ids:
        race_id = str(race_id).strip()
        if not re.fullmatch(r'\d{12}', race_id):
            warn_print(f'race_idの形式が12桁数字ではありません: {race_id}')
            continue
        if race_id in seen:
            debug_print(f'重複race_idをスキップします: {race_id}')
            continue
        seen.add(race_id)
        normalized.append(race_id)
    return normalized


def is_jra_race_id(race_id):
    return str(race_id)[4:6] in JRA_PLACE_CODES


def market_from_race_id(race_id):
    return '中央' if is_jra_race_id(race_id) else '地方'


def build_netkeiba_url(race_id, page_kind, market=None):
    if market is None:
        market = market_from_race_id(race_id)
    if page_kind not in {'shutuba', 'result'}:
        raise ValueError(f'page_kindが不正です: {page_kind}')
    host = 'race.netkeiba.com' if market == '中央' else 'nar.netkeiba.com'
    return f'https://{host}/race/{page_kind}.html?race_id={race_id}'


def fetch_html(session, url):
    debug_print(f'GET {url}')
    response = session.get(url, timeout=30)
    response.raise_for_status()
    if not response.encoding or response.encoding.lower() in {'iso-8859-1', 'ascii'}:
        response.encoding = response.apparent_encoding
    return response.text


def soup_from_html(html):
    return BeautifulSoup(html, 'lxml')


def parse_japanese_date(text):
    match = re.search(r'(\d{4})年(\d{1,2})月(\d{1,2})日', text)
    if not match:
        return ''
    year = int(match.group(1))
    month = int(match.group(2))
    day = int(match.group(3))
    return f'{year:04d}-{month:02d}-{day:02d}'


def yen_to_int(value):
    text = clean_text(value)
    digits = re.sub(r'[^0-9]', '', text)
    return int(digits) if digits else ''


def numeric_sort_value(value, default=9999):
    text = clean_text(value)
    if re.fullmatch(r'\d+', text):
        return int(text)
    return default


def split_cell_tokens(cell):
    if cell is None:
        return []
    return [clean_text(token) for token in cell.stripped_strings if clean_text(token)]


def combo_key(values):
    return tuple(sorted(str(v).strip() for v in values if str(v).strip()))

# ============================================================
# 4. netkeibaパーサー
# ============================================================
def parse_course_data(race_data01):
    course_type = ''
    distance_m = ''
    course_note = ''
    weather = ''
    going = ''

    match = re.search(r'/\s*([^/]*?)(\d{3,4})m\s*(?:\((.*?)\))?', race_data01)
    if match:
        course_type = clean_text(match.group(1)).replace(' ', '')
        distance_m = int(match.group(2))
        course_note = clean_text(match.group(3))

    match = re.search(r'天候\s*[:：]\s*([^/\s]+)', race_data01)
    if match:
        weather = clean_text(match.group(1))

    match = re.search(r'馬場\s*[:：]\s*([^/\s]+)', race_data01)
    if match:
        going = clean_text(match.group(1))

    return course_type, distance_m, course_note, weather, going


def parse_race_meta(soup, race_id, market):
    title = meta_content(soup, 'og:title') or (cell_text(soup.title) if soup.title else '')

    race_name = cell_text(soup.select_one('.RaceName'))
    if not race_name:
        race_name = re.split(r'\s+(?:出馬表|結果・払戻)', title)[0].strip()

    race_num_text = cell_text(soup.select_one('.RaceNum'))
    race_no_match = re.search(r'(\d+)R', race_num_text or title)
    race_no = int(race_no_match.group(1)) if race_no_match else ''

    race_date = parse_japanese_date(title)

    place = ''
    place_match = re.search(r'\d{4}年\d{1,2}月\d{1,2}日\s+(.+?)(\d+)R', title)
    if place_match:
        place = clean_text(place_match.group(1))
    if not place and race_id[4:6] in JRA_PLACE_CODES:
        place = JRA_PLACE_CODES[race_id[4:6]]

    race_data01 = cell_text(soup.select_one('.RaceData01'))
    race_data02 = cell_text(soup.select_one('.RaceData02'))
    course_type, distance_m, course_note, weather, going = parse_course_data(race_data01)

    head_count = ''
    head_count_match = re.search(r'(\d+)頭', race_data02)
    if head_count_match:
        head_count = int(head_count_match.group(1))

    return {
        'race_id': race_id,
        '開催区分': market,
        '日付': race_date,
        '場': place,
        'R': race_no,
        'レース名': race_name,
        'コース種別': course_type,
        '距離m': distance_m,
        'コース補足': course_note,
        '天候': weather,
        '馬場': going,
        '頭数': head_count,
    }


def build_prediction_row(meta):
    row = {header: '' for header in PREDICTION_HEADERS}
    for header in PREDICTION_HEADERS:
        if header in meta:
            row[header] = meta[header]
    return row


def parse_result_finishers(soup, race_id):
    table = soup.select_one('#All_Result_Table') or soup.select_one('table.ResultRefund')
    if table is None:
        warn_print(f'結果テーブルが見つかりません。race_id={race_id}')
        return {}

    finishers = {}
    for tr in table.find_all('tr'):
        cells = tr.find_all(['td', 'th'], recursive=False)
        if not cells or cells[0].name == 'th':
            continue

        values = [cell_text(cell) for cell in cells]
        if len(values) < 4:
            continue

        rank_text = values[0]
        rank_match = re.match(r'^(\d+)', rank_text)
        if not rank_match:
            continue

        rank = int(rank_match.group(1))
        if rank < 1 or rank > 10:
            continue

        horse_number = clean_text(values[2])
        horse_name = clean_text(values[3])
        if not horse_name:
            continue

        if rank not in finishers:
            finishers[rank] = []
        finishers[rank].append({
            'rank': rank,
            'horse_number': horse_number,
            'horse_name': horse_name,
        })

    debug_print(f'race_id={race_id} 結果上位取得: {sum(len(v) for v in finishers.values())}頭')
    return finishers


def parse_payout_entries(soup, race_id):
    entries = []
    tables = soup.select('table.Payout_Detail_Table')
    if not tables:
        warn_print(f'払戻テーブルが見つかりません。race_id={race_id}')
        return entries

    for table in tables:
        for tr in table.find_all('tr'):
            cells = tr.find_all(['th', 'td'], recursive=False)
            if len(cells) < 3:
                continue

            ticket_type = cell_text(cells[0])
            result_tokens = split_cell_tokens(cells[1])
            payout_tokens = split_cell_tokens(cells[2])

            if not ticket_type or not result_tokens or not payout_tokens:
                continue

            if ticket_type in {'単勝', '複勝'}:
                group_size = 1
            elif len(payout_tokens) > 1:
                group_size = max(1, len(result_tokens) // len(payout_tokens))
            else:
                group_size = len(result_tokens)

            for index, payout_text in enumerate(payout_tokens):
                start = index * group_size
                end = start + group_size
                result_group = result_tokens[start:end]
                if not result_group:
                    result_group = result_tokens

                entries.append({
                    '券種': ticket_type,
                    '組番': result_group,
                    '払戻円': yen_to_int(payout_text),
                })

    debug_print(f'race_id={race_id} 払戻取得: {len(entries)}件')
    return entries


def get_first_payout(entries, ticket_type):
    for entry in entries:
        if entry['券種'] == ticket_type:
            return entry['払戻円']
    return ''


def get_combo_payout(entries, ticket_type, numbers, ordered=False):
    target_numbers = [str(number).strip() for number in numbers if str(number).strip()]
    if not target_numbers:
        return ''

    for entry in entries:
        if entry['券種'] != ticket_type:
            continue
        entry_numbers = [str(number).strip() for number in entry['組番'] if str(number).strip()]
        if ordered:
            if entry_numbers == target_numbers:
                return entry['払戻円']
        else:
            if combo_key(entry_numbers) == combo_key(target_numbers):
                return entry['払戻円']
    return ''


def first_finisher(finishers, rank):
    rows = finishers.get(rank) or []
    return rows[0] if rows else {'horse_number': '', 'horse_name': ''}


def build_result_row(meta, finishers, payout_entries):
    row = {header: '' for header in RESULT_HEADERS}

    for header in ['race_id', '開催区分', '日付', '場']:
        row[header] = meta.get(header, '')

    for rank in range(1, 11):
        rows = finishers.get(rank) or []
        row[f'{rank}着'] = ' / '.join(item['horse_name'] for item in rows)

    top1 = first_finisher(finishers, 1)
    top2 = first_finisher(finishers, 2)
    top3 = first_finisher(finishers, 3)
    top1_no = top1['horse_number']
    top2_no = top2['horse_number']
    top3_no = top3['horse_number']

    row['単勝'] = get_combo_payout(payout_entries, '単勝', [top1_no], ordered=False) or get_first_payout(payout_entries, '単勝')
    row['複勝（1着）'] = get_combo_payout(payout_entries, '複勝', [top1_no], ordered=False)
    row['複勝（2着）'] = get_combo_payout(payout_entries, '複勝', [top2_no], ordered=False)
    row['複勝（3着）'] = get_combo_payout(payout_entries, '複勝', [top3_no], ordered=False)
    row['枠連'] = get_first_payout(payout_entries, '枠連')
    row['馬連'] = get_combo_payout(payout_entries, '馬連', [top1_no, top2_no], ordered=False) or get_first_payout(payout_entries, '馬連')
    row['ワイド（1-2着）'] = get_combo_payout(payout_entries, 'ワイド', [top1_no, top2_no], ordered=False)
    row['ワイド（1-3着）'] = get_combo_payout(payout_entries, 'ワイド', [top1_no, top3_no], ordered=False)
    row['ワイド（2-3着）'] = get_combo_payout(payout_entries, 'ワイド', [top2_no, top3_no], ordered=False)
    row['馬単'] = get_combo_payout(payout_entries, '馬単', [top1_no, top2_no], ordered=True) or get_first_payout(payout_entries, '馬単')
    row['3連複'] = get_combo_payout(payout_entries, '3連複', [top1_no, top2_no, top3_no], ordered=False) or get_first_payout(payout_entries, '3連複')
    row['3連単'] = get_combo_payout(payout_entries, '3連単', [top1_no, top2_no, top3_no], ordered=True) or get_first_payout(payout_entries, '3連単')

    return row


def fetch_and_parse_one_race(session, race_id):
    market = market_from_race_id(race_id)
    shutuba_url = build_netkeiba_url(race_id, 'shutuba', market)
    result_url = build_netkeiba_url(race_id, 'result', market)

    shutuba_html = fetch_html(session, shutuba_url)
    shutuba_soup = soup_from_html(shutuba_html)
    meta = parse_race_meta(shutuba_soup, race_id, market)
    prediction_row = build_prediction_row(meta)

    result_row = None
    try:
        result_html = fetch_html(session, result_url)
        result_soup = soup_from_html(result_html)
        result_meta = parse_race_meta(result_soup, race_id, market)
        meta_for_result = {**meta, **{k: v for k, v in result_meta.items() if v != ''}}
        finishers = parse_result_finishers(result_soup, race_id)
        payout_entries = parse_payout_entries(result_soup, race_id)

        if finishers:
            result_row = build_result_row(meta_for_result, finishers, payout_entries)
        else:
            warn_print(f'結果が未取得のため、結果入力は更新しません。race_id={race_id}')
    except Exception as exc:
        warn_print(f'結果ページの取得または解析に失敗しました。race_id={race_id}, error={repr(exc)}')

    debug_print(f'race_id={race_id} 予想入力用基本情報: {prediction_row}')
    if result_row:
        debug_print(f'race_id={race_id} 結果入力用レコード: {result_row}')

    return prediction_row, result_row

# ============================================================
# 5. Google Sheets操作
# ============================================================
def authorize_gspread():
    if auth is None or default is None:
        raise RuntimeError('Google Colab上で実行してください。ローカル実行の場合は認証処理を別途用意してください。')
    auth.authenticate_user()
    credentials, _ = default()
    return gspread.authorize(credentials)


def get_or_create_worksheet(spreadsheet, title, rows=1000, cols=30):
    try:
        ws = spreadsheet.worksheet(title)
        debug_print(f'既存シートを使用します: {title}')
        return ws
    except gspread.WorksheetNotFound:
        debug_print(f'シートを作成します: {title}')
        return spreadsheet.add_worksheet(title=title, rows=rows, cols=cols)


def safe_resize(ws, rows, cols):
    try:
        ws.resize(rows=rows, cols=cols)
    except Exception as exc:
        warn_print(f'シートサイズ変更に失敗しました。sheet={ws.title}, error={repr(exc)}')


def safe_clear(ws):
    try:
        ws.clear()
    except Exception as exc:
        warn_print(f'シートクリアに失敗しました。sheet={ws.title}, error={repr(exc)}')


def safe_freeze(ws, rows=1):
    try:
        ws.freeze(rows=rows)
    except Exception as exc:
        warn_print(f'固定行設定に失敗しました。sheet={ws.title}, error={repr(exc)}')


def safe_format(ws, cell_range, fmt):
    try:
        ws.format(cell_range, fmt)
    except Exception as exc:
        warn_print(f'書式設定に失敗しました。sheet={ws.title}, range={cell_range}, error={repr(exc)}')


def get_existing_rows(ws):
    values = ws.get_all_values()
    if not values:
        return []

    headers = [clean_text(header) for header in values[0]]
    rows = []
    for raw_row in values[1:]:
        row = {}
        for index, header in enumerate(headers):
            if not header:
                continue
            row[header] = raw_row[index] if index < len(raw_row) else ''
        if any(clean_text(value) for value in row.values()):
            rows.append(row)
    return rows


def rows_to_values(headers, rows):
    values = [headers]
    for row in rows:
        values.append([row.get(header, '') for header in headers])
    return values


def write_table_sheet(ws, headers, rows):
    target_rows = max(len(rows) + 1, 2)
    safe_clear(ws)
    safe_resize(ws, rows=target_rows + 20, cols=len(headers))
    ws.update('A1', rows_to_values(headers, rows), value_input_option='USER_ENTERED')
    safe_freeze(ws, rows=1)
    safe_format(ws, f'A1:{column_letter(len(headers))}1', {
        'textFormat': {'bold': True, 'foregroundColor': {'red': 1, 'green': 1, 'blue': 1}},
        'backgroundColor': {'red': 0.15, 'green': 0.35, 'blue': 0.55},
        'horizontalAlignment': 'CENTER',
    })


def column_letter(n):
    result = ''
    while n:
        n, rem = divmod(n - 1, 26)
        result = chr(65 + rem) + result
    return result


def sort_prediction_rows(rows):
    return sorted(
        rows,
        key=lambda row: (
            clean_text(row.get('日付', '')),
            clean_text(row.get('開催区分', '')),
            clean_text(row.get('場', '')),
            numeric_sort_value(row.get('R', '')),
            clean_text(row.get('race_id', '')),
        ),
    )


def sort_result_rows(rows):
    return sorted(
        rows,
        key=lambda row: (
            clean_text(row.get('日付', '')),
            clean_text(row.get('開催区分', '')),
            clean_text(row.get('場', '')),
            clean_text(row.get('race_id', '')),
        ),
    )


def upsert_prediction_sheet(ws, new_rows):
    existing_rows = get_existing_rows(ws)
    manual_by_race_id = {}

    for row in existing_rows:
        race_id = clean_text(row.get('race_id', ''))
        if not race_id:
            continue
        manual_by_race_id[race_id] = {
            header: row.get(header, '')
            for header in PREDICTION_RANK_HEADERS
        }

    new_ids = {clean_text(row.get('race_id', '')) for row in new_rows}
    for row in new_rows:
        race_id = clean_text(row.get('race_id', ''))
        for header in PREDICTION_RANK_HEADERS:
            row[header] = manual_by_race_id.get(race_id, {}).get(header, row.get(header, ''))

    kept_rows = []
    for existing in existing_rows:
        race_id = clean_text(existing.get('race_id', ''))
        if not race_id or race_id in new_ids:
            continue
        kept_rows.append({header: existing.get(header, '') for header in PREDICTION_HEADERS})

    merged_rows = sort_prediction_rows(kept_rows + new_rows)
    write_table_sheet(ws, PREDICTION_HEADERS, merged_rows)
    debug_print(f'予想入力を更新しました。対象race_id数={len(new_ids)}, 既存保持={len(kept_rows)}, 合計={len(merged_rows)}')
    return merged_rows


def replace_result_rows_by_race_id(ws, new_rows, replace_race_ids):
    existing_rows = get_existing_rows(ws)
    replace_race_ids = {clean_text(race_id) for race_id in replace_race_ids if clean_text(race_id)}

    kept_rows = []
    for existing in existing_rows:
        race_id = clean_text(existing.get('race_id', ''))
        if not race_id or race_id in replace_race_ids:
            continue
        kept_rows.append({header: existing.get(header, '') for header in RESULT_HEADERS})

    merged_rows = sort_result_rows(kept_rows + new_rows)
    write_table_sheet(ws, RESULT_HEADERS, merged_rows)
    debug_print(f'結果入力を更新しました。置換race_id数={len(replace_race_ids)}, 追加={len(new_rows)}, 既存保持={len(kept_rows)}, 合計={len(merged_rows)}')
    return merged_rows

# ============================================================
# 6. メイン処理
# ============================================================
def main():
    race_ids = normalize_race_ids(RACE_ID_LIST)
    if not race_ids:
        raise ValueError('RACE_ID_LISTに有効なrace_idがありません。')

    debug_print(f'処理対象race_id: {race_ids}')

    session = requests.Session()
    session.headers.update({
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                      '(KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36',
        'Accept-Language': 'ja,en-US;q=0.9,en;q=0.8',
    })

    prediction_rows = []
    result_rows = []
    result_replace_ids = set()

    for index, race_id in enumerate(race_ids, start=1):
        print(f'[{index}/{len(race_ids)}] race_id={race_id} を処理します。')
        try:
            prediction_row, result_row = fetch_and_parse_one_race(session, race_id)
            prediction_rows.append(prediction_row)

            if result_row:
                result_rows.append(result_row)
                result_replace_ids.add(race_id)
            else:
                warn_print(f'結果入力の既存レコードは置換しません。race_id={race_id}')

        except Exception as exc:
            warn_print(f'race_idの処理に失敗しました。race_id={race_id}, error={repr(exc)}')

        if index < len(race_ids):
            time.sleep(REQUEST_INTERVAL_SEC)

    if not prediction_rows:
        raise RuntimeError('出馬表から基本レース情報を1件も取得できませんでした。')

    spreadsheet_id = parse_spreadsheet_id(SPREADSHEET_URL)
    gc = authorize_gspread()
    spreadsheet = gc.open_by_key(spreadsheet_id)

    ws_prediction = get_or_create_worksheet(spreadsheet, SHEET_PREDICTION, rows=1000, cols=len(PREDICTION_HEADERS))
    ws_result = get_or_create_worksheet(spreadsheet, SHEET_RESULT, rows=1000, cols=len(RESULT_HEADERS))

    upsert_prediction_sheet(ws_prediction, prediction_rows)

    if result_replace_ids:
        replace_result_rows_by_race_id(ws_result, result_rows, result_replace_ids)
    else:
        debug_print('結果入力は置換対象がないため更新をスキップします。')

    print('完了しました。')
    print(f'予想入力 取得または更新: {len(prediction_rows)}レース')
    print(f'結果入力 取得または更新: {len(result_rows)}レース')


if __name__ == '__main__':
    main()
